# Chapter 46: Motion Estimation

## Chapter objective

The goal of this notebook is to estimate how image regions move between two frames.
We will use a small **educational patch-matching demo**: pick a patch in frame 1,
search for the most similar patch in frame 2, and interpret the displacement as a
motion vector.

This is intentionally a controlled synthetic motion-estimation example. It is useful
for learning the core ideas behind correspondence and local motion, but it is **not**
a full optical-flow system.


## Beginner-friendly intuition

Here are the main ideas we will use:

- **Frame 1** is the earlier image.
- **Frame 2** is the later image after the scene has moved.
- A **local patch** is a small square neighborhood around one pixel.
- A **search window** says where we are willing to look in frame 2.
- A **matching cost** measures how different two patches are.
- The **motion vector** is the displacement `(dx, dy)` from the patch location in frame 1
  to the best match in frame 2.
  In this notebook, positive `dx` means rightward motion and positive `dy` means downward motion in image coordinates.
- Candidate patches near image borders may be skipped because a full patch cannot be extracted safely.

We will stay in a simple synthetic setting with known ground-truth translation. That lets
us inspect the cost surface, compare predicted motion with the true answer, and see both
success cases and failure cases clearly.


In [ ]:
import math
import time

import matplotlib.pyplot as plt
import torch
from matplotlib.patches import Rectangle

torch.manual_seed(7)
torch.set_num_threads(1)

device = torch.device("cpu")
plt.rcParams.update(
    {
        "figure.figsize": (6, 4),
        "image.cmap": "gray",
        "axes.grid": False,
        "font.size": 11,
    }
)

print(f"Using device: {device}")
print("Random seed fixed to 7 for reproducibility.")


In [ ]:
def create_synthetic_frame(height=80, width=120, noise_level=0.0, kind="textured"):
    """Create a small grayscale frame with either rich texture or an ambiguous pattern."""
    ys = torch.linspace(-1.0, 1.0, height, device=device)
    xs = torch.linspace(-1.0, 1.0, width, device=device)
    yy, xx = torch.meshgrid(ys, xs, indexing="ij")

    if kind == "textured":
        base = (
            0.30
            + 0.18 * torch.sin(3.0 * math.pi * xx + 0.6 * torch.cos(2.0 * math.pi * yy))
            + 0.14 * torch.cos(4.0 * math.pi * yy)
            + 0.12 * torch.sin(5.0 * math.pi * (xx + 0.35 * yy))
        )
        checker = 0.10 * torch.sin(10.0 * math.pi * xx) * torch.cos(8.0 * math.pi * yy)
        blob1 = 0.30 * torch.exp(-((xx + 0.35) ** 2 + (yy - 0.20) ** 2) / 0.05)
        blob2 = 0.25 * torch.exp(-((xx - 0.25) ** 2 + (yy + 0.25) ** 2) / 0.03)
        edge = 0.12 * ((xx > -0.1) & (yy < 0.15)).float()
        frame = base + checker + blob1 + blob2 + edge
    elif kind == "repeated":
        stripes = 0.45 + 0.22 * torch.sin(6.0 * math.pi * xx)
        weak_y_variation = 0.04 * torch.cos(2.0 * math.pi * yy)
        frame = stripes + weak_y_variation
    elif kind == "low_texture":
        frame = 0.5 + 0.03 * xx + 0.02 * yy
    else:
        raise ValueError(f"Unknown frame kind: {kind}")

    if noise_level > 0:
        frame = frame + noise_level * torch.randn_like(frame)

    frame = frame - frame.min()
    frame = frame / frame.max().clamp_min(1e-6)
    return frame.clamp(0.0, 1.0)


def shift_image(image, dx, dy, fill_value=0.0):
    """Translate an image by integer pixels. Positive dx moves right, positive dy moves down."""
    shifted = torch.full_like(image, fill_value)
    height, width = image.shape

    if abs(dx) >= width or abs(dy) >= height:
        return shifted

    if dx >= 0:
        src_x = slice(0, width - dx)
        dst_x = slice(dx, width)
    else:
        src_x = slice(-dx, width)
        dst_x = slice(0, width + dx)

    if dy >= 0:
        src_y = slice(0, height - dy)
        dst_y = slice(dy, height)
    else:
        src_y = slice(-dy, height)
        dst_y = slice(0, height + dy)

    shifted[dst_y, dst_x] = image[src_y, src_x]
    return shifted


def extract_patch(image, center_y, center_x, patch_size):
    """Extract an odd-sized square patch centered at (center_y, center_x)."""
    if patch_size % 2 == 0:
        raise ValueError("patch_size must be odd")

    radius = patch_size // 2
    y0, y1 = center_y - radius, center_y + radius + 1
    x0, x1 = center_x - radius, center_x + radius + 1

    if y0 < 0 or x0 < 0 or y1 > image.shape[0] or x1 > image.shape[1]:
        raise ValueError("Patch exceeds image bounds")

    return image[y0:y1, x0:x1]


def compute_ssd(patch_a, patch_b):
    """Sum of squared differences."""
    return float(torch.sum((patch_a - patch_b) ** 2).item())


def compute_sad(patch_a, patch_b):
    """Sum of absolute differences."""
    return float(torch.sum(torch.abs(patch_a - patch_b)).item())


def search_best_match(frame1, frame2, center_y, center_x, patch_size, search_radius, cost="ssd"):
    """Search over a local window in frame 2 and return the best integer-pixel match."""
    query_patch = extract_patch(frame1, center_y, center_x, patch_size)
    cost_name = cost.lower()

    if cost_name == "ssd":
        cost_fn = compute_ssd
    elif cost_name == "sad":
        cost_fn = compute_sad
    else:
        raise ValueError("cost must be either 'ssd' or 'sad'")

    cost_map = torch.full((2 * search_radius + 1, 2 * search_radius + 1), float("nan"))
    best_cost = float("inf")
    best_center = (center_y, center_x)
    best_patch = query_patch.clone()
    comparisons = 0
    start_time = time.perf_counter()

    for row, dy in enumerate(range(-search_radius, search_radius + 1)):
        for col, dx in enumerate(range(-search_radius, search_radius + 1)):
            candidate_y = center_y + dy
            candidate_x = center_x + dx

            try:
                candidate_patch = extract_patch(frame2, candidate_y, candidate_x, patch_size)
            except ValueError:
                continue

            comparisons += 1
            current_cost = cost_fn(query_patch, candidate_patch)
            cost_map[row, col] = current_cost

            if current_cost < best_cost:
                best_cost = current_cost
                best_center = (candidate_y, candidate_x)
                best_patch = candidate_patch.clone()

    runtime_ms = 1000.0 * (time.perf_counter() - start_time)
    radius = patch_size // 2
    search_window = {
        "x0": max(0, center_x - search_radius - radius),
        "x1": min(frame2.shape[1], center_x + search_radius + radius + 1),
        "y0": max(0, center_y - search_radius - radius),
        "y1": min(frame2.shape[0], center_y + search_radius + radius + 1),
    }

    return {
        "query_center": (center_y, center_x),
        "best_center": best_center,
        "motion": (best_center[1] - center_x, best_center[0] - center_y),
        "best_cost": best_cost,
        "comparisons": comparisons,
        "runtime_ms": runtime_ms,
        "cost_map": cost_map,
        "query_patch": query_patch,
        "best_patch": best_patch,
        "search_window": search_window,
        "patch_size": patch_size,
        "search_radius": search_radius,
        "cost_name": cost_name,
    }


def estimate_sparse_motion_field(frame1, frame2, sample_points, patch_size, search_radius, cost="ssd"):
    """Estimate motion only at a small set of sample points."""
    estimates = []
    for center_y, center_x in sample_points:
        match = search_best_match(
            frame1,
            frame2,
            center_y=center_y,
            center_x=center_x,
            patch_size=patch_size,
            search_radius=search_radius,
            cost=cost,
        )
        match["point"] = (center_y, center_x)
        estimates.append(match)
    return estimates


def compute_endpoint_error(predicted_motion, ground_truth_motion):
    """Euclidean distance between predicted and true motion vectors."""
    pred_dx, pred_dy = predicted_motion
    gt_dx, gt_dy = ground_truth_motion
    return float(math.sqrt((pred_dx - gt_dx) ** 2 + (pred_dy - gt_dy) ** 2))


def make_translated_pair(height=80, width=120, gt_motion=(5, 3), noise_level=0.0, kind="textured"):
    """Build a pair of frames that share known ground-truth translation."""
    clean_frame1 = create_synthetic_frame(height=height, width=width, kind=kind, noise_level=0.0)
    clean_frame2 = shift_image(clean_frame1, dx=gt_motion[0], dy=gt_motion[1], fill_value=0.0)

    if noise_level > 0:
        frame1 = (clean_frame1 + noise_level * torch.randn_like(clean_frame1)).clamp(0.0, 1.0)
        frame2 = (clean_frame2 + noise_level * torch.randn_like(clean_frame2)).clamp(0.0, 1.0)
    else:
        frame1 = clean_frame1.clone()
        frame2 = clean_frame2.clone()

    return frame1, frame2, clean_frame1, clean_frame2


## Synthetic data setup

We will generate a textured grayscale frame directly in PyTorch, then create frame 2 by
shifting the content by a known translation. The ground-truth motion vector is stored so we
can compare predictions against the correct answer.


In [ ]:
height, width = 80, 120
ground_truth_motion = (5, 3)  # (dx, dy)
noise_level = 0.02

frame1, frame2, clean_frame1, clean_frame2 = make_translated_pair(
    height=height,
    width=width,
    gt_motion=ground_truth_motion,
    noise_level=noise_level,
    kind="textured",
)

print(f"Frame size: {height} x {width}")
print(f"Known ground-truth motion: dx={ground_truth_motion[0]}, dy={ground_truth_motion[1]}")
print(f"Noise level used for this demo pair: {noise_level}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(frame1.numpy(), vmin=0.0, vmax=1.0)
axes[0].set_title("Frame 1")
axes[1].imshow(frame2.numpy(), vmin=0.0, vmax=1.0)
axes[1].set_title("Frame 2")

for ax in axes:
    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.suptitle("Synthetic translation example")
plt.tight_layout()
plt.show()


## Single-patch motion estimation

We now pick one patch from frame 1, search inside a local window in frame 2, and inspect
the matching cost surface. This is the core patch-matching idea in its simplest form.


In [ ]:
query_center = (34, 42)  # (y, x)
patch_size = 11
search_radius = 8

single_match_ssd = search_best_match(
    frame1,
    frame2,
    center_y=query_center[0],
    center_x=query_center[1],
    patch_size=patch_size,
    search_radius=search_radius,
    cost="ssd",
)
single_match_sad = search_best_match(
    frame1,
    frame2,
    center_y=query_center[0],
    center_x=query_center[1],
    patch_size=patch_size,
    search_radius=search_radius,
    cost="sad",
)

predicted_motion = single_match_ssd["motion"]
endpoint_error = compute_endpoint_error(predicted_motion, ground_truth_motion)
search_window = single_match_ssd["search_window"]
best_center = single_match_ssd["best_center"]
radius = patch_size // 2

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(frame1.numpy(), vmin=0.0, vmax=1.0)
axes[0].add_patch(
    Rectangle(
        (query_center[1] - radius, query_center[0] - radius),
        patch_size,
        patch_size,
        fill=False,
        edgecolor="cyan",
        linewidth=2,
    )
)
axes[0].set_title("Query patch in frame 1")

axes[1].imshow(
    frame2[search_window["y0"] : search_window["y1"], search_window["x0"] : search_window["x1"]].numpy(),
    vmin=0.0,
    vmax=1.0,
)
axes[1].set_title("Search window in frame 2")
axes[1].add_patch(
    Rectangle(
        (
            best_center[1] - radius - search_window["x0"],
            best_center[0] - radius - search_window["y0"],
        ),
        patch_size,
        patch_size,
        fill=False,
        edgecolor="lime",
        linewidth=2,
    )
)

cost_map = single_match_ssd["cost_map"]
heatmap = axes[2].imshow(cost_map.numpy(), cmap="magma")
axes[2].scatter(
    single_match_ssd["motion"][0] + search_radius,
    single_match_ssd["motion"][1] + search_radius,
    c="cyan",
    s=80,
    label="best",
)
axes[2].set_title("Matching cost heatmap (SSD)")
axes[2].set_xlabel("candidate dx index")
axes[2].set_ylabel("candidate dy index")
axes[2].legend(loc="upper right")
plt.colorbar(heatmap, ax=axes[2], fraction=0.046)

axes[3].imshow(single_match_ssd["best_patch"].numpy(), vmin=0.0, vmax=1.0)
axes[3].set_title("Best matching patch")

for ax in axes[:2]:
    ax.set_xlabel("x")
    ax.set_ylabel("y")
for ax in axes[2:]:
    ax.set_xticks([0, search_radius, 2 * search_radius])
    ax.set_xticklabels([-search_radius, 0, search_radius])
    ax.set_yticks([0, search_radius, 2 * search_radius])
    ax.set_yticklabels([-search_radius, 0, search_radius])

plt.tight_layout()
plt.show()

print(f"Predicted motion with SSD: {single_match_ssd['motion']}")
print(f"Predicted motion with SAD: {single_match_sad['motion']}")
print(f"Ground-truth motion:       {ground_truth_motion}")
print(f"Endpoint error:            {endpoint_error:.3f}")
print(f"Candidate comparisons:     {single_match_ssd['comparisons']}")
print(f"Search runtime (ms):       {single_match_ssd['runtime_ms']:.2f}")


## Sparse motion field and validation metrics

Instead of estimating motion at every pixel, we can sample a small grid of points and
compute a **sparse motion field**. This keeps the demo lightweight while still letting us
measure accuracy over multiple patches. Here, **correct match ratio** means exact integer-vector
recovery of the known synthetic motion, so it is a controlled validation metric rather than a
general optical-flow benchmark.


In [ ]:
sample_points = [(y, x) for y in [18, 30, 42, 54] for x in [22, 42, 62, 82]]
sparse_matches = estimate_sparse_motion_field(
    frame1,
    frame2,
    sample_points=sample_points,
    patch_size=patch_size,
    search_radius=search_radius,
    cost="ssd",
)

endpoint_errors = [compute_endpoint_error(match["motion"], ground_truth_motion) for match in sparse_matches]
correct_matches = [match["motion"] == ground_truth_motion for match in sparse_matches]
correct_match_ratio = sum(correct_matches) / len(correct_matches)
total_comparisons = sum(match["comparisons"] for match in sparse_matches)
total_runtime_ms = sum(match["runtime_ms"] for match in sparse_matches)

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(frame1.numpy(), vmin=0.0, vmax=1.0)
xs = [point[1] for point in sample_points]
ys = [point[0] for point in sample_points]
us = [match["motion"][0] for match in sparse_matches]
vs = [match["motion"][1] for match in sparse_matches]
colors = ["tab:green" if ok else "tab:red" for ok in correct_matches]

ax.quiver(xs, ys, us, vs, color=colors, angles="xy", scale_units="xy", scale=1)
ax.scatter(xs, ys, c=colors, s=25)
ax.set_xlim(0, width)
ax.set_ylim(height, 0)
ax.set_title("Sparse motion vectors overlaid on frame 1")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.show()

print(f"Number of sampled patches: {len(sample_points)}")
print(f"Predicted motion vector example: {sparse_matches[0]['motion']}")
print(f"Ground-truth motion vector:      {ground_truth_motion}")
print(f"Mean endpoint error:             {sum(endpoint_errors) / len(endpoint_errors):.3f}")
print(f"Max endpoint error:              {max(endpoint_errors):.3f}")
print(f"Correct match ratio:             {correct_match_ratio:.3f}")
print(f"Total candidate comparisons:     {total_comparisons}")
print(f"Approximate total runtime (ms):  {total_runtime_ms:.2f}")


## Parameter study

Patch matching depends strongly on three practical choices:

- **Patch size**: larger patches carry more context, but also assume the whole patch moves together.
- **Search radius**: too small and the true match may be outside the window.
- **Noise level**: noisier frames make matching costs less reliable.

We will sweep one parameter at a time and track **correct match ratio** and **mean endpoint error**.
In this notebook, correct match ratio again means exact integer-vector recovery of the known synthetic displacement.


In [ ]:
def evaluate_configuration(patch_size_value=11, search_radius_value=8, noise_level_value=0.02):
    frame_a, frame_b, _, _ = make_translated_pair(
        height=height,
        width=width,
        gt_motion=ground_truth_motion,
        noise_level=noise_level_value,
        kind="textured",
    )
    matches = estimate_sparse_motion_field(
        frame_a,
        frame_b,
        sample_points=sample_points,
        patch_size=patch_size_value,
        search_radius=search_radius_value,
        cost="ssd",
    )
    errors = [compute_endpoint_error(match["motion"], ground_truth_motion) for match in matches]
    ratio = sum(match["motion"] == ground_truth_motion for match in matches) / len(matches)
    return {
        "mean_epe": sum(errors) / len(errors),
        "correct_ratio": ratio,
    }


patch_sizes = [7, 11, 15]
search_radii = [4, 6, 8, 10]
noise_levels = [0.00, 0.04, 0.08, 0.12]

patch_results = [evaluate_configuration(patch_size_value=value, search_radius_value=8, noise_level_value=0.08) for value in patch_sizes]
radius_results = [evaluate_configuration(patch_size_value=11, search_radius_value=value, noise_level_value=0.02) for value in search_radii]
noise_results = [evaluate_configuration(patch_size_value=11, search_radius_value=8, noise_level_value=value) for value in noise_levels]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(patch_sizes, [result["correct_ratio"] for result in patch_results], marker="o", linewidth=2)
axes[0].set_title("Patch size sweep")
axes[0].set_xlabel("patch size")
axes[0].set_ylabel("correct match ratio")
axes[0].set_ylim(-0.05, 1.05)

axes[1].plot(search_radii, [result["correct_ratio"] for result in radius_results], marker="o", linewidth=2)
axes[1].set_title("Search radius sweep")
axes[1].set_xlabel("search radius")
axes[1].set_ylabel("correct match ratio")
axes[1].set_ylim(-0.05, 1.05)

axes[2].plot(noise_levels, [result["correct_ratio"] for result in noise_results], marker="o", linewidth=2)
axes[2].set_title("Noise sweep")
axes[2].set_xlabel("noise level")
axes[2].set_ylabel("correct match ratio")
axes[2].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

print("Patch size sweep:")
for value, result in zip(patch_sizes, patch_results):
    print(f"  patch_size={value:>2} -> correct_ratio={result['correct_ratio']:.3f}, mean_epe={result['mean_epe']:.3f}")

print("\nSearch radius sweep:")
for value, result in zip(search_radii, radius_results):
    print(f"  search_radius={value:>2} -> correct_ratio={result['correct_ratio']:.3f}, mean_epe={result['mean_epe']:.3f}")

print("\nNoise sweep:")
for value, result in zip(noise_levels, noise_results):
    print(f"  noise_level={value:>4.2f} -> correct_ratio={result['correct_ratio']:.3f}, mean_epe={result['mean_epe']:.3f}")


In this controlled sweep, too-small search radii can exclude the true motion entirely, while larger radii usually improve recovery at the cost of more candidate comparisons.
Moderate patch sizes are often more stable than tiny noisy patches because they carry more structure, and higher noise weakens the distinctiveness of the matching minimum.


## Failure cases

Patch matching works best when the true displacement is inside the search window and the
local patch has distinctive texture. The next cell demonstrates two common failure modes:

1. **Search radius too small** for the real motion.
2. **Repeated-pattern ambiguity**, where many candidate locations look almost identical.


In [ ]:
small_radius_match = search_best_match(
    frame1,
    frame2,
    center_y=query_center[0],
    center_x=query_center[1],
    patch_size=patch_size,
    search_radius=3,
    cost="ssd",
)

repeated_gt_motion = (6, 2)
repeated_frame1, repeated_frame2, _, _ = make_translated_pair(
    height=height,
    width=width,
    gt_motion=repeated_gt_motion,
    noise_level=0.0,
    kind="repeated",
)
repeated_center = (32, 40)
repeated_match = search_best_match(
    repeated_frame1,
    repeated_frame2,
    center_y=repeated_center[0],
    center_x=repeated_center[1],
    patch_size=11,
    search_radius=8,
    cost="ssd",
)

repeated_cost_map = repeated_match["cost_map"]
flat_costs = repeated_cost_map.reshape(-1)
valid_mask = ~torch.isnan(flat_costs)
valid_indices = torch.arange(flat_costs.numel())[valid_mask]
valid_costs = flat_costs[valid_mask]
sorted_positions = torch.argsort(valid_costs)[:5]

print("Failure case 1: search radius too small")
print(f"  predicted motion: {small_radius_match['motion']}")
print(f"  ground truth:     {ground_truth_motion}")
print(f"  endpoint error:   {compute_endpoint_error(small_radius_match['motion'], ground_truth_motion):.3f}")

print("\nFailure case 2: repeated-pattern ambiguity")
print(f"  predicted motion: {repeated_match['motion']}")
print(f"  ground truth:     {repeated_gt_motion}")
print(f"  endpoint error:   {compute_endpoint_error(repeated_match['motion'], repeated_gt_motion):.3f}")
print("  five lowest SSD candidates (dx, dy, cost):")
for position in sorted_positions.tolist():
    flat_index = valid_indices[position].item()
    row = flat_index // repeated_cost_map.shape[1]
    col = flat_index % repeated_cost_map.shape[1]
    dx = col - repeated_match['search_radius']
    dy = row - repeated_match['search_radius']
    cost_value = float(valid_costs[position].item())
    print(f"    ({dx:>2}, {dy:>2}) -> {cost_value:.6f}")

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

axes[0, 0].imshow(frame1.numpy(), vmin=0.0, vmax=1.0)
axes[0, 0].add_patch(
    Rectangle(
        (query_center[1] - radius, query_center[0] - radius),
        patch_size,
        patch_size,
        fill=False,
        edgecolor="cyan",
        linewidth=2,
    )
)
axes[0, 0].set_title("Failure 1: query patch")

axes[0, 1].imshow(frame2.numpy(), vmin=0.0, vmax=1.0)
axes[0, 1].add_patch(
    Rectangle(
        (query_center[1] - 3 - radius, query_center[0] - 3 - radius),
        2 * 3 + patch_size,
        2 * 3 + patch_size,
        fill=False,
        edgecolor="orange",
        linewidth=2,
    )
)
axes[0, 1].set_title("Search window misses true motion")

heatmap_small = axes[0, 2].imshow(small_radius_match["cost_map"].numpy(), cmap="magma")
axes[0, 2].scatter(
    small_radius_match["motion"][0] + 3,
    small_radius_match["motion"][1] + 3,
    c="cyan",
    s=80,
)
axes[0, 2].set_title("Cost map with small radius")
plt.colorbar(heatmap_small, ax=axes[0, 2], fraction=0.046)

axes[1, 0].imshow(repeated_frame1.numpy(), vmin=0.0, vmax=1.0)
axes[1, 0].add_patch(
    Rectangle(
        (repeated_center[1] - 5, repeated_center[0] - 5),
        11,
        11,
        fill=False,
        edgecolor="cyan",
        linewidth=2,
    )
)
axes[1, 0].set_title("Failure 2: repeated patch")

axes[1, 1].imshow(repeated_frame2.numpy(), vmin=0.0, vmax=1.0)
axes[1, 1].set_title("Repeated-pattern frame 2")

heatmap_repeat = axes[1, 2].imshow(repeated_match["cost_map"].numpy(), cmap="magma")
axes[1, 2].scatter(
    repeated_match["motion"][0] + repeated_match["search_radius"],
    repeated_match["motion"][1] + repeated_match["search_radius"],
    c="cyan",
    s=80,
)
axes[1, 2].set_title("Many similar minima")
plt.colorbar(heatmap_repeat, ax=axes[1, 2], fraction=0.046)

for ax in [axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]]:
    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

for ax, local_radius in [(axes[0, 2], 3), (axes[1, 2], repeated_match["search_radius"])]:
    ax.set_xticks([0, local_radius, 2 * local_radius])
    ax.set_xticklabels([-local_radius, 0, local_radius])
    ax.set_yticks([0, local_radius, 2 * local_radius])
    ax.set_yticklabels([-local_radius, 0, local_radius])
    ax.set_xlabel("candidate dx")
    ax.set_ylabel("candidate dy")

plt.tight_layout()
plt.show()


In the repeated-pattern example, multiple similar low-cost matches make correspondence unstable because the best match is not unique.


## Limitations

This educational patch-matching demo is intentionally simple. Important limitations:

- It only does **integer-pixel matching**, not subpixel refinement.
- It assumes **brightness consistency** between frames.
- It struggles in **low-texture regions**, **repeated patterns**, **occlusion**, and near **motion boundaries**.
- It searches only a small local window and estimates a sparse set of points.
- It is **not a modern learned optical-flow system**.


## Reproducibility notes

- The random seed is fixed at `7`.
- The notebook is written to run on **CPU**.
- No external image files are required.
- All frames are generated synthetically inside the notebook.

That makes the notebook easy to run end-to-end and easy to modify for further experiments.
